# Simple Iron Powder Reconstruction Example

This notebook demonstrates a complete workflow for:
1. Loading iron powder data
2. Processing with instrument response
3. Applying Poisson sampling
4. Frame overlap and reconstruction
5. Bragg edge fitting with nbragg
6. **Bragg edge position tracking using sigmoid fitting method**

## Study 1: Single Frame vs Pulse Duration
Test reconstruction quality as a function of pulse duration (10-1000 µs)

## Study 2: Multiple Frames with 50 µs Pulse
Test reconstruction quality with 1-30 random frames at pulse_duration=50 µs

## Study 3: 8 Frames, 50 µs Pulse - Random Seed Sweep
Test how random seed affects reconstruction with 8 frames at 50 µs pulse

## Study 4: Noise Level Sweep for Best Seed
Optimize Wiener filter noise_power using the best seed from Study 3

All studies track:
- Reconstruction quality (χ²/dof, R²)
- nbragg fit quality (reduced χ²)
- Sample thickness and cellulose fraction
- **Bragg edge position around 2.4 Å** using sigmoid fitting (robust for pulse-broadened edges)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from frame_overlap import Data, Reconstruct, Analysis
from scipy.optimize import curve_fit
from scipy.special import erf
import warnings
import sys
import os
from contextlib import contextmanager

# Suppress verbose warnings
warnings.filterwarnings('ignore', category=UserWarning, module='nbragg')
warnings.filterwarnings('ignore', category=RuntimeWarning, module='pandas')

%matplotlib inline

In [ ]:
def extract_bragg_edge_position(result, target_wavelength=2.4, search_width=0.5):
    """
    Extract Bragg edge position using sigmoid fitting method.
    Best for pulse-broadened edges.
    
    Parameters
    ----------
    result : lmfit.ModelResult
        The nbragg fit result containing best_fit transmission
    target_wavelength : float
        Target wavelength to search near (Angstroms)
    search_width : float
        Search range ± around target (Angstroms), default 0.5
    
    Returns
    -------
    edge_position : float
        Wavelength at Bragg edge (Angstroms)
    edge_error : float
        Estimated uncertainty in edge position (Angstroms)
    """
    # Get wavelength and transmission from result
    wavelength = result.userkws["wl"]
    transmission = result.best_fit
    
    # Create region mask
    mask = (wavelength >= target_wavelength - search_width) & \
           (wavelength <= target_wavelength + search_width)
    wl_region = wavelength[mask]
    trans_region = transmission[mask]
    
    if len(wl_region) < 5:
        return np.nan, np.nan
    
    # Define edge model: baseline + step*erf((x-x0)/width)
    def edge_model(x, baseline, step, x0, width):
        return baseline + step * 0.5 * (1 + erf((x - x0) / width))
    
    # Initial guess
    baseline_guess = np.mean(trans_region[-3:])  # Right side (higher wavelength)
    step_guess = np.mean(trans_region[:3]) - baseline_guess  # Jump height (negative)
    x0_guess = target_wavelength
    width_guess = 0.05
    
    try:
        popt, pcov = curve_fit(
            edge_model, wl_region, trans_region,
            p0=[baseline_guess, step_guess, x0_guess, width_guess],
            bounds=([0, -np.inf, target_wavelength-search_width, 0.001],
                    [1, 0, target_wavelength+search_width, search_width])
        )
        edge_position = popt[2]
        edge_error = np.sqrt(pcov[2, 2]) if pcov[2, 2] > 0 else np.nan
        
        return edge_position, edge_error
    except:
        # Fallback to midpoint method if sigmoid fitting fails
        # Estimate baseline (before edge) and post-edge level
        baseline = np.percentile(trans_region, 80)
        post_edge = np.percentile(trans_region, 20)
        
        # Find midpoint crossing
        midpoint = (baseline + post_edge) / 2
        
        # Find where transmission crosses midpoint
        crossings = np.where(np.diff(np.sign(trans_region - midpoint)))[0]
        
        if len(crossings) > 0:
            idx = crossings[0]
            # Linear interpolation for sub-pixel accuracy
            edge_position = np.interp(midpoint, 
                                      [trans_region[idx+1], trans_region[idx]],
                                      [wl_region[idx+1], wl_region[idx]])
            
            # Error estimate from scatter
            edge_error = np.std(trans_region) / np.abs(baseline - post_edge) * \
                         np.mean(np.diff(wl_region))
        else:
            edge_position, edge_error = np.nan, np.nan
        
        return edge_position, edge_error

In [ ]:
@contextmanager
def suppress_stdout():
    """Context manager to suppress stdout temporarily."""
    with open(os.devnull, 'w') as devnull:
        old_stdout = sys.stdout
        sys.stdout = devnull
        try:
            yield
        finally:
            sys.stdout = old_stdout

### Test the Bragg edge extraction function

This section creates a test reconstruction and verifies the edge extraction function works correctly before running the parameter sweeps.

In [ ]:
# Create test data for verifying edge extraction function
print("Creating test reconstruction...")

data = Data(
    signal_file='iron_powder.csv',
    openbeam_file='openbeam.csv',
    flux=5e6,
    duration=0.5,
    freq=20
)
pulse_dur = 50
data.convolute_response(pulse_duration=pulse_dur)
data.poisson_sample(flux=1e6, freq=20, measurement_time=8.0)
data.overlap(kernel=1, total_time=50, mode='equal')  # Single frame

# Reconstruct
test_recon = Reconstruct(data, tmin=None, tmax=None)
test_recon.filter(kind='wiener', noise_power=1.0)

# nbragg fit
analysis = Analysis(
    xs='iron_cellulose_fixed_response',
    vary_background=False,
    vary_response=True,
    vary_weights=True,
    response="square_jorgensen",
    pulse_duration=pulse_dur
)
analysis.set_params(temp={'vary': False})
test_result = analysis.fit(test_recon, wlmin=1.0, wlmax=5.0)

print("✓ Test data created successfully")

In [ ]:
# Test the improved Bragg edge extraction function
print("Testing Sigmoid-based Bragg edge extraction")
print("=" * 60)

# Test 1: Basic functionality
print("\nTest 1: Basic extraction")
print("-" * 60)

edge_pos, edge_err = extract_bragg_edge_position(test_result, target_wavelength=2.4, search_width=0.5)
print(f"✓ Edge position: {edge_pos:.4f} ± {edge_err:.4f} Å")
print(f"  Target: 2.4 Å, Deviation: {abs(edge_pos - 2.4):.4f} Å")

# Visualize the fit
wavelength = test_result.userkws["wl"]
transmission = test_result.best_fit

mask = (wavelength >= edge_pos - 0.5) & (wavelength <= edge_pos + 0.5)
wl_plot = wavelength[mask]
trans_plot = transmission[mask]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Plot 1: Transmission data and fitted edge
ax1.plot(wl_plot, trans_plot, 'o', label='Data', alpha=0.6, markersize=3)
ax1.axvline(edge_pos, color='red', linestyle='--', label=f'Edge: {edge_pos:.4f} Å', linewidth=2)
ax1.axvline(2.4, color='gray', linestyle=':', label='Target: 2.4 Å', alpha=0.5)
ax1.set_xlabel('Wavelength (Å)', fontsize=11)
ax1.set_ylabel('Transmission', fontsize=11)
ax1.set_title('Bragg Edge Detection', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: Derivative to show edge sharpness
derivative = -np.gradient(trans_plot, wl_plot)
ax2.plot(wl_plot, derivative, '-', color='blue', linewidth=1.5)
ax2.axvline(edge_pos, color='red', linestyle='--', linewidth=2)
ax2.set_xlabel('Wavelength (Å)', fontsize=11)
ax2.set_ylabel('-dT/dλ', fontsize=11)
ax2.set_title('Transmission Derivative', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Test 2: Different search widths
print(f"\n\nTest 2: Robustness to search width")
print("-" * 60)
for width in [0.3, 0.5, 0.8, 1.0]:
    pos, err = extract_bragg_edge_position(test_result, target_wavelength=2.4, search_width=width)
    if np.isnan(pos):
        print(f"  Width {width:.1f} Å: ✗ Failed")
    else:
        print(f"  Width {width:.1f} Å: ✓ {pos:.4f} ± {err:.4f} Å")

# Test 3: Different target wavelengths (iron edges)
print(f"\n\nTest 3: Multiple iron Bragg edges")
print("-" * 60)
for target_wl in [2.0, 2.4, 2.9]:
    pos, err = extract_bragg_edge_position(test_result, target_wavelength=target_wl, search_width=0.5)
    if np.isnan(pos):
        print(f"  Target {target_wl:.1f} Å: ✗ No edge found")
    else:
        print(f"  Target {target_wl:.1f} Å: ✓ {pos:.4f} ± {err:.4f} Å")

print(f"\n{'='*60}")
print(f"✓ TESTS COMPLETE - Sigmoid fitting method is working")
print(f"{'='*60}")

## Study 1: Single Frame - Pulse Duration Sweep

Test how pulse duration affects reconstruction with a single frame (no overlap)

In [ ]:
# Pulse duration values to test (µs)
pulse_durations = np.arange(10, 1000, 10)
results_pulse = []

for pulse_dur in pulse_durations:
    print(f"Processing pulse_duration = {pulse_dur} µs...", end=' ')
    
    try:
        with suppress_stdout():
            # Create data object
            data = Data(
                signal_file='iron_powder.csv',
                openbeam_file='openbeam.csv',
                flux=5e6,
                duration=0.5,
                freq=20
            )
            
            # Process pipeline
            data.convolute_response(pulse_duration=pulse_dur)
            data.poisson_sample(flux=1e6, freq=20, measurement_time=8.0)
            data.overlap(kernel=1, total_time=50, mode='equal')
            
            # Reconstruct
            recon = Reconstruct(data, tmin=None, tmax=None)
            recon.filter(kind='wiener', noise_power=1.0)
            
            # Get reconstruction statistics
            stats = recon.get_statistics()
            
            # nbragg fit
            analysis = Analysis(
                xs='iron_cellulose_fixed_response',
                vary_background=False,
                vary_response=True,
                vary_weights=True,
                response="square_jorgensen",
                pulse_duration=pulse_dur
            )
            analysis.set_params(temp={'vary': False})
            result = analysis.fit(recon, wlmin=1.0, wlmax=5.0)
        
        # Extract parameters
        thickness = result.params['thickness'].value
        thickness_err = result.params['thickness'].stderr if result.params['thickness'].stderr else 0
        cellulose = result.params['cellulose'].value
        cellulose_err = result.params['cellulose'].stderr if result.params['cellulose'].stderr else 0
        edge_pos, edge_err = extract_bragg_edge_position(result, target_wavelength=2.4, search_width=0.5)
        
        # Store results
        results_pulse.append({
            'pulse_duration': pulse_dur,
            'chi2_per_dof': stats['chi2_per_dof'],
            'r_squared': stats['r_squared'],
            'nbragg_redchi': result.redchi,
            'thickness': thickness,
            'thickness_err': thickness_err,
            'cellulose': cellulose,
            'cellulose_err': cellulose_err,
            'bragg_edge_pos': edge_pos,
            'bragg_edge_err': edge_err
        })
        
        print(f"✓ χ²={result.redchi:.3f}")
        
    except Exception as e:
        print(f"✗ {e}")
        continue

# Convert to DataFrame
results_pulse_df = pd.DataFrame(results_pulse)
print(f"\n✓ Sweep complete! Processed {len(results_pulse_df)} pulse durations")
print(results_pulse_df.head())

In [ ]:
# Plot Study 1 results
fig, axes = plt.subplots(4, 1, figsize=(6, 10), sharex=True)

# Thickness
axes[0].plot(results_pulse_df['pulse_duration'], results_pulse_df['thickness'], color="royalblue")
axes[0].fill_between(results_pulse_df['pulse_duration'], 
                     results_pulse_df["thickness"] - results_pulse_df["thickness_err"],
                     results_pulse_df["thickness"] + results_pulse_df["thickness_err"],
                     alpha=0.5, color="0.5", zorder=-1)
axes[0].axhline(1.977588, ls="--", color="0.2", zorder=-2)
axes[0].set_ylim(1.9, 2)
axes[0].set_ylabel('Thickness [cm]', fontsize=14)

# Cellulose
axes[1].plot(results_pulse_df['pulse_duration'], results_pulse_df['cellulose']*100, color="royalblue")
axes[1].fill_between(results_pulse_df['pulse_duration'],
                     results_pulse_df["cellulose"]*100 - results_pulse_df["cellulose_err"]*100,
                     results_pulse_df["cellulose"]*100 + results_pulse_df["cellulose_err"]*100,
                     alpha=0.5, color="0.5", zorder=-1)
axes[1].axhline(100*0.0208, ls="--", color="0.2", zorder=-2)
axes[1].set_ylim(1, 6)
axes[1].set_ylabel('Cellulose fraction [%]\n', fontsize=14)

# Bragg edge position
axes[2].plot(results_pulse_df['pulse_duration'], results_pulse_df['bragg_edge_pos'], color="royalblue")
axes[2].fill_between(results_pulse_df['pulse_duration'],
                     results_pulse_df["bragg_edge_pos"] - results_pulse_df["bragg_edge_err"],
                     results_pulse_df["bragg_edge_pos"] + results_pulse_df["bragg_edge_err"],
                     alpha=0.5, color="0.5", zorder=-1)
axes[2].set_ylabel('Bragg Edge Position [Å]\n', fontsize=14)

# nbragg reduced chi2
axes[3].plot(results_pulse_df['pulse_duration'], results_pulse_df['nbragg_redchi'], '-', 
            color='indianred', linewidth=1, markersize=5)
axes[3].set_ylabel('Reduced χ² (nbragg)\n', fontsize=14)
axes[3].set_xlabel('Pulse Duration (µs)', fontsize=14)

plt.tight_layout()
plt.show()

## Study 2: Multiple Frames with 50 µs Pulse

Test reconstruction quality with varying number of frames at fixed pulse_duration=50 µs

In [ ]:
# Number of frames to test
n_frames_values = range(1, 31)
pulse_dur = 50  # µs
results_frames = []

for n_frames in n_frames_values:
    print(f"Processing n_frames = {n_frames}...", end=' ')
    
    try:
        with suppress_stdout():
            # Create data object
            data = Data(
                signal_file='iron_powder.csv',
                openbeam_file='openbeam.csv',
                flux=5e6,
                duration=0.5,
                freq=20
            )
            
            # Process pipeline
            data.convolute_response(pulse_duration=pulse_dur)
            data.poisson_sample(flux=1e6, freq=20, measurement_time=8.0)
            data.overlap(kernel=n_frames, total_time=50, mode='random', kernel_seed=42)
            
            # Reconstruct
            recon = Reconstruct(data, tmin=None, tmax=None)
            recon.filter(kind='wiener', noise_power=1.0)
            
            # Get reconstruction statistics
            stats = recon.get_statistics()
            
            # nbragg fit
            analysis = Analysis(
                xs='iron_cellulose_fixed_response',
                vary_background=False,
                vary_response=True,
                vary_weights=True,
                response="square_jorgensen",
                pulse_duration=pulse_dur
            )
            analysis.set_params(temp={'vary': False})
            result = analysis.fit(recon, wlmin=1.0, wlmax=2.5)
        
        # Extract parameters
        thickness = result.params['thickness'].value
        thickness_err = result.params['thickness'].stderr if result.params['thickness'].stderr else 0
        cellulose = result.params['cellulose'].value
        cellulose_err = result.params['cellulose'].stderr if result.params['cellulose'].stderr else 0
        edge_pos, edge_err = extract_bragg_edge_position(result, target_wavelength=2.4, search_width=0.5)
        
        # Store results
        results_frames.append({
            'n_frames': n_frames,
            'chi2_per_dof': stats['chi2_per_dof'],
            'r_squared': stats['r_squared'],
            'nbragg_redchi': result.redchi,
            'thickness': thickness,
            'thickness_err': thickness_err,
            'cellulose': cellulose,
            'cellulose_err': cellulose_err,
            'bragg_edge_pos': edge_pos,
            'bragg_edge_err': edge_err
        })
        
        print(f"✓ χ²={result.redchi:.3f}")
        
    except Exception as e:
        print(f"✗ {e}")
        continue

# Convert to DataFrame
results_frames_df = pd.DataFrame(results_frames)
print(f"\n✓ Sweep complete! Processed {len(results_frames_df)} frame configurations")
print(results_frames_df.head())

In [ ]:
# Plot Study 2 results
fig, axes = plt.subplots(4, 1, figsize=(6, 10), sharex=True)

# Thickness
axes[0].plot(results_frames_df['n_frames'], results_frames_df['thickness'], color="royalblue")
axes[0].fill_between(results_frames_df['n_frames'],
                     results_frames_df["thickness"] - results_frames_df["thickness_err"],
                     results_frames_df["thickness"] + results_frames_df["thickness_err"],
                     alpha=0.5, color="0.5", zorder=-1)
axes[0].axhline(1.977588, ls="--", color="0.2", zorder=-2)
axes[0].set_ylabel('Thickness [cm]', fontsize=14)

# Cellulose
axes[1].plot(results_frames_df['n_frames'], results_frames_df['cellulose']*100, color="royalblue")
axes[1].fill_between(results_frames_df['n_frames'],
                     results_frames_df["cellulose"]*100 - results_frames_df["cellulose_err"]*100,
                     results_frames_df["cellulose"]*100 + results_frames_df["cellulose_err"]*100,
                     alpha=0.5, color="0.5", zorder=-1)
axes[1].axhline(100*0.0208, ls="--", color="0.2", zorder=-2)
axes[1].set_ylabel('Cellulose fraction [%]\n', fontsize=14)

# Bragg edge position
axes[2].plot(results_frames_df['n_frames'], results_frames_df['bragg_edge_pos'], color="royalblue")
axes[2].fill_between(results_frames_df['n_frames'],
                     results_frames_df["bragg_edge_pos"] - results_frames_df["bragg_edge_err"],
                     results_frames_df["bragg_edge_pos"] + results_frames_df["bragg_edge_err"],
                     alpha=0.5, color="0.5", zorder=-1)
axes[2].set_ylabel('Bragg Edge Position [Å]\n', fontsize=14)

# nbragg reduced chi2
axes[3].plot(results_frames_df['n_frames'], results_frames_df['nbragg_redchi'], '-', 
            color='indianred', linewidth=1, markersize=5)
axes[3].set_ylabel('Reduced χ² (nbragg)\n', fontsize=14)
axes[3].set_xlabel('Number of random overlap frames', fontsize=14)

plt.tight_layout()
plt.show()

## Study 3: 8 Frames, 50 µs Pulse - Random Seed Sweep

Test how random seed affects reconstruction with fixed n_frames=8 and pulse_duration=50 µs

In [ ]:
# Random seeds to test
random_seeds = range(10, 60)
n_frames_fixed = 8
pulse_dur_fixed = 50  # µs
results_seeds = []

for seed in random_seeds:
    print(f"Processing seed = {seed}...", end=' ')
    
    try:
        with suppress_stdout():
            # Create data object
            data = Data(
                signal_file='iron_powder.csv',
                openbeam_file='openbeam.csv',
                flux=5e6,
                duration=0.5,
                freq=20
            )
            
            # Process pipeline
            data.convolute_response(pulse_duration=pulse_dur_fixed)
            data.poisson_sample(flux=1e6, freq=20, measurement_time=8.0)
            data.overlap(kernel=n_frames_fixed, total_time=50, mode='random', kernel_seed=seed)
            
            # Reconstruct
            recon = Reconstruct(data, tmin=None, tmax=None)
            recon.filter(kind='wiener', noise_power=1.0)
            
            # Get reconstruction statistics
            stats = recon.get_statistics()
            
            # nbragg fit
            analysis = Analysis(
                xs='iron_cellulose_fixed_response',
                vary_background=False,
                vary_response=True,
                vary_weights=True,
                response="square_jorgensen",
                pulse_duration=pulse_dur_fixed
            )
            analysis.set_params(temp={'vary': False})
            result = analysis.fit(recon, wlmin=1.0, wlmax=2.5)
        
        # Extract parameters
        thickness = result.params['thickness'].value
        thickness_err = result.params['thickness'].stderr if result.params['thickness'].stderr else 0
        cellulose = result.params['cellulose'].value
        cellulose_err = result.params['cellulose'].stderr if result.params['cellulose'].stderr else 0
        edge_pos, edge_err = extract_bragg_edge_position(result, target_wavelength=2.4, search_width=0.5)
        
        # Store results
        results_seeds.append({
            'seed': seed,
            'chi2_per_dof': stats['chi2_per_dof'],
            'r_squared': stats['r_squared'],
            'nbragg_redchi': result.redchi,
            'thickness': thickness,
            'thickness_err': thickness_err,
            'cellulose': cellulose,
            'cellulose_err': cellulose_err,
            'bragg_edge_pos': edge_pos,
            'bragg_edge_err': edge_err
        })
        
        print(f"✓ χ²={result.redchi:.3f}")
        
    except Exception as e:
        print(f"✗ {e}")
        continue

# Convert to DataFrame
results_seeds_df = pd.DataFrame(results_seeds)

# Find best seed
best_seed_idx = results_seeds_df['nbragg_redchi'].idxmin()
best_seed = results_seeds_df.loc[best_seed_idx, 'seed']

print(f"\n✓ Sweep complete! Processed {len(results_seeds_df)} random seeds")
print(f"✓ Best seed: {best_seed} with χ² = {results_seeds_df.loc[best_seed_idx, 'nbragg_redchi']:.4f}")
print(results_seeds_df.head())

In [ ]:
# Plot Study 3 results
fig, axes = plt.subplots(4, 1, figsize=(6, 10), sharex=True)

# Thickness
axes[0].plot(results_seeds_df['seed'], results_seeds_df['thickness'], color="royalblue")
axes[0].fill_between(results_seeds_df['seed'],
                     results_seeds_df["thickness"] - results_seeds_df["thickness_err"],
                     results_seeds_df["thickness"] + results_seeds_df["thickness_err"],
                     alpha=0.5, color="0.5", zorder=-1)
axes[0].axhline(1.977588, ls="--", color="0.2", zorder=-2)
axes[0].set_ylim(1.85, 2.05)
axes[0].set_ylabel('Thickness [cm]', fontsize=14)

# Cellulose
axes[1].plot(results_seeds_df['seed'], results_seeds_df['cellulose']*100, color="royalblue")
axes[1].fill_between(results_seeds_df['seed'],
                     results_seeds_df["cellulose"]*100 - results_seeds_df["cellulose_err"]*100,
                     results_seeds_df["cellulose"]*100 + results_seeds_df["cellulose_err"]*100,
                     alpha=0.5, color="0.5", zorder=-1)
axes[1].axhline(100*0.0208, ls="--", color="0.2", zorder=-2)
axes[1].set_ylim(0, 10)
axes[1].set_ylabel('Cellulose fraction [%]\n', fontsize=14)

# Bragg edge position
axes[2].plot(results_seeds_df['seed'], results_seeds_df['bragg_edge_pos'], color="royalblue")
axes[2].fill_between(results_seeds_df['seed'],
                     results_seeds_df["bragg_edge_pos"] - results_seeds_df["bragg_edge_err"],
                     results_seeds_df["bragg_edge_pos"] + results_seeds_df["bragg_edge_err"],
                     alpha=0.5, color="0.5", zorder=-1)
axes[2].set_ylabel('Bragg Edge Position [Å]\n', fontsize=14)

# nbragg reduced chi2
axes[3].plot(results_seeds_df['seed'], results_seeds_df['nbragg_redchi'], '-', 
            color='indianred', linewidth=1, markersize=5)
axes[3].set_ylim(0, 100)
axes[3].set_ylabel('Reduced χ² (nbragg)\n', fontsize=14)
axes[3].set_xlabel('Random Seed', fontsize=14)

plt.tight_layout()
plt.show()

## Study 4: Noise Level Sweep for Best Seed

Test reconstruction quality with different noise_power values using the best seed from Study 3

In [ ]:
# Noise power values to test
noise_powers = np.logspace(-2, 1, 30)
results_noise = []

for noise_power in noise_powers:
    print(f"Processing noise_power = {noise_power:.3f}...", end=' ')
    
    try:
        with suppress_stdout():
            # Create data object
            data = Data(
                signal_file='iron_powder.csv',
                openbeam_file='openbeam.csv',
                flux=5e6,
                duration=0.5,
                freq=20
            )
            
            # Process pipeline
            data.convolute_response(pulse_duration=pulse_dur_fixed)
            data.poisson_sample(flux=1e6, freq=20, measurement_time=8.0)
            data.overlap(kernel=n_frames_fixed, total_time=50, mode='random', kernel_seed=best_seed)
            
            # Reconstruct with varying noise power
            recon = Reconstruct(data, tmin=None, tmax=None)
            recon.filter(kind='wiener', noise_power=noise_power)
            
            # Get reconstruction statistics
            stats = recon.get_statistics()
            
            # nbragg fit
            analysis = Analysis(
                xs='iron_cellulose_fixed_response',
                vary_background=False,
                vary_response=True,
                vary_weights=True,
                response="square_jorgensen",
                pulse_duration=pulse_dur_fixed
            )
            analysis.set_params(temp={'vary': False})
            result = analysis.fit(recon, wlmin=1.0, wlmax=2.5)
        
        # Extract parameters
        thickness = result.params['thickness'].value
        thickness_err = result.params['thickness'].stderr if result.params['thickness'].stderr else 0
        cellulose = result.params['cellulose'].value
        cellulose_err = result.params['cellulose'].stderr if result.params['cellulose'].stderr else 0
        edge_pos, edge_err = extract_bragg_edge_position(result, target_wavelength=2.4, search_width=0.5)
        
        # Store results
        results_noise.append({
            'noise_power': noise_power,
            'chi2_per_dof': stats['chi2_per_dof'],
            'r_squared': stats['r_squared'],
            'nbragg_redchi': result.redchi,
            'thickness': thickness,
            'thickness_err': thickness_err,
            'cellulose': cellulose,
            'cellulose_err': cellulose_err,
            'bragg_edge_pos': edge_pos,
            'bragg_edge_err': edge_err
        })
        
        print(f"✓ χ²={result.redchi:.3f}")
        
    except Exception as e:
        print(f"✗ {e}")
        continue

# Convert to DataFrame
results_noise_df = pd.DataFrame(results_noise)

# Find best noise_power
best_noise_idx = results_noise_df['nbragg_redchi'].idxmin()
best_noise_power = results_noise_df.loc[best_noise_idx, 'noise_power']

print(f"\n✓ Sweep complete! Processed {len(results_noise_df)} noise power values")
print(f"✓ Best noise_power: {best_noise_power:.4f} with χ² = {results_noise_df.loc[best_noise_idx, 'nbragg_redchi']:.4f}")
print(results_noise_df.head())

In [ ]:
# Plot Study 4 results
fig, axes = plt.subplots(4, 1, figsize=(6, 10), sharex=True)

# Thickness
axes[0].plot(results_noise_df['noise_power'], results_noise_df['thickness'], color="royalblue")
axes[0].fill_between(results_noise_df['noise_power'],
                     results_noise_df["thickness"] - results_noise_df["thickness_err"],
                     results_noise_df["thickness"] + results_noise_df["thickness_err"],
                     alpha=0.5, color="0.5", zorder=-1)
axes[0].axhline(1.977588, ls="--", color="0.2", zorder=-2)
axes[0].set_ylabel('Thickness [cm]', fontsize=14)
axes[0].set_xscale('log')

# Cellulose
axes[1].plot(results_noise_df['noise_power'], results_noise_df['cellulose']*100, color="royalblue")
axes[1].fill_between(results_noise_df['noise_power'],
                     results_noise_df["cellulose"]*100 - results_noise_df["cellulose_err"]*100,
                     results_noise_df["cellulose"]*100 + results_noise_df["cellulose_err"]*100,
                     alpha=0.5, color="0.5", zorder=-1)
axes[1].axhline(100*0.0208, ls="--", color="0.2", zorder=-2)
axes[1].set_ylabel('Cellulose fraction [%]\n', fontsize=14)
axes[1].set_xscale('log')

# Bragg edge position
axes[2].plot(results_noise_df['noise_power'], results_noise_df['bragg_edge_pos'], color="royalblue")
axes[2].fill_between(results_noise_df['noise_power'],
                     results_noise_df["bragg_edge_pos"] - results_noise_df["bragg_edge_err"],
                     results_noise_df["bragg_edge_pos"] + results_noise_df["bragg_edge_err"],
                     alpha=0.5, color="0.5", zorder=-1)
axes[2].set_ylabel('Bragg Edge Position [Å]\n', fontsize=14)
axes[2].set_xscale('log')

# nbragg reduced chi2
axes[3].plot(results_noise_df['noise_power'], results_noise_df['nbragg_redchi'], '-', 
            color='indianred', linewidth=1, markersize=5)
axes[3].set_ylabel('Reduced χ² (nbragg)\n', fontsize=14)
axes[3].set_xlabel('Wiener Filter Noise Power', fontsize=14)
axes[3].set_xscale('log')

plt.tight_layout()
plt.show()

## Summary

This notebook demonstrated:

### Study 1: Single Frame - Pulse Duration Effects
- Tested pulse durations: 10-1000 µs in 10 µs steps
- Single frame (no overlap)
- Shows how instrument resolution affects reconstruction and fitting
- Tracks Bragg edge position around 2.4 Å using sigmoid fitting

### Study 2: Multiple Frames at 50 µs Pulse
- Tested 1-30 random frames
- Fixed pulse duration: 50 µs, seed: 42
- Shows trade-off between frame overlap and reconstruction quality
- Tracks Bragg edge position variation with number of frames

### Study 3: 8 Frames, 50 µs Pulse - Random Seed Sweep
- Tested 50 different random seeds (10-59)
- Fixed: n_frames=8, pulse_duration=50 µs
- Shows how random frame placement affects results
- Identifies best seed for lowest nbragg χ²
- Tracks Bragg edge position sensitivity to seed

### Study 4: Noise Level Sweep for Best Seed
- Tested noise_power from 0.01 to 10 (logarithmic spacing)
- Uses best seed from Study 3
- Fixed: n_frames=8, pulse_duration=50 µs
- Optimizes Wiener filter regularization parameter
- Tracks Bragg edge position stability vs noise power

### nbragg Fit Settings (Used in All Studies)
- **Model**: iron_cellulose_fixed_response
- **vary_background**: False
- **vary_response**: True
- **vary_weights**: True
- **vary_temperature**: False (never varied)
- **response**: square_jorgensen
- **Wavelength range**: 1-5 Å (Study 1), 1-2.5 Å (Studies 2-4)

### Bragg Edge Analysis Method
- **Primary method**: Sigmoid/error function fitting
  - Models edge as smooth transition: `T(λ) = baseline + step × erf((λ - λ₀)/width)`
  - Optimal for pulse-broadened edges
  - Provides robust position (λ₀) and uncertainty estimates
- **Fallback method**: Midpoint crossing
  - Used when sigmoid fitting fails
  - Very robust for noisy data
- Tracked edge position and uncertainty across all parameter sweeps
- Provides direct measure of reconstruction quality on physical observable

### Key Observations
- Longer pulse durations blur the Bragg edges but may improve statistics
- More frames generally improve reconstruction but increase complexity
- Random seed selection can significantly impact reconstruction quality
- Wiener filter noise_power needs optimization for best results
- Bragg edge position is a robust metric for reconstruction quality
- Sigmoid fitting handles pulse-broadened edges better than derivative methods
- The iron_cellulose_fixed_response model accounts for cellulose impurities

### Next Steps
- Compare different reconstruction methods (wiener, fobi, lucy-richardson)
- Test with real experimental data
- Investigate edge-specific analysis for multiple Bragg edges
- Optimize measurement_time for target precision
- Explore alternative edge detection methods (Savitzky-Golay, cumulative distribution)